# LLM-as-Judge: Automated Evaluation with Language Models

As open-ended generation became the dominant LLM use case, metrics requiring reference answers became bottlenecks. LLM-as-judge emerged as the practical solution: use a capable model to evaluate responses against a rubric, approximating human preference at scale.

## Why Reference-Free Evaluation Matters

For instruction following, coding assistance, and multi-turn conversation, there is no single correct reference answer. Collecting human annotations at scale costs thousands of dollars per benchmark.

LLM-as-judge offers a middle path: structured rubric evaluation that correlates with human preference at roughly 80-90% agreement, running at API cost.

Key papers:
- **MT-Bench** (Zheng et al. 2023): GPT-4 as judge achieves >80% agreement with human raters on multi-turn conversations
- **Alpaca Eval**: uses GPT-4 to rank instruction-following outputs against reference responses
- **Constitutional AI** (Anthropic 2022): model self-critique as part of the RLHF pipeline

## Bias Patterns in LLM Judges

LLM judges introduce their own failure modes:

**Verbosity bias**: judges prefer longer responses even when a shorter answer is more accurate.

**Positional bias**: systematic preference for whichever response was presented first (or second). Can be 15-20 percentage points.

**Self-enhancement bias**: a model used as judge rates outputs similar to its own style more favorably.

**Sycophancy toward confident tone**: authoritative-sounding responses rated higher regardless of accuracy.

In [ ]:
import random
from dataclasses import dataclass

@dataclass
class JudgeResult:
    winner: str
    score_a: float
    score_b: float
    reasoning: str
    judge_model: str

def make_mock_judge(verbosity_bias=0.3, positional_bias=0.15, seed=42):
    rng = random.Random(seed)
    def judge(prompt, response_a, response_b):
        base_a = min(9.0, 5.0 + len(response_a)/200)
        base_b = min(9.0, 5.0 + len(response_b)/200)
        vb_a = verbosity_bias * (len(response_a) - 150) / 150
        vb_b = verbosity_bias * (len(response_b) - 150) / 150
        score_a = min(10, max(1, base_a + vb_a + positional_bias + rng.gauss(0, 0.3)))
        score_b = min(10, max(1, base_b + vb_b + rng.gauss(0, 0.3)))
        winner = 'tie' if abs(score_a - score_b) < 0.5 else ('a' if score_a > score_b else 'b')
        return JudgeResult(winner=winner, score_a=round(score_a,2), score_b=round(score_b,2),
                           reasoning='Response A was more detailed.' if score_a >= score_b else 'Response B was more detailed.',
                           judge_model='mock-v1')
    return judge

judge = make_mock_judge()
prompt = 'What is gradient descent?'
short_correct = 'Gradient descent iteratively moves parameters in the direction of steepest loss decrease.'
long_verbose = ('Gradient descent is a fundamental optimization algorithm. It computes the gradient of the loss '
                'function with respect to all parameters, then updates each parameter by moving a small step '
                'opposite to the gradient. The step size is controlled by the learning rate. There are several '
                'variants: batch, stochastic, and mini-batch gradient descent.')

result = judge(prompt, short_correct, long_verbose)
print(f'Winner: {result.winner}')
print(f'Score A (short, correct): {result.score_a}')
print(f'Score B (long, verbose):  {result.score_b}')
print(f'Reasoning: {result.reasoning}')

## Detecting and Correcting Positional Bias

The standard mitigation is **position swapping**: run each evaluation twice with A and B swapped. If the verdict flips based solely on position, the result is unreliable.

In [ ]:
def detect_positional_bias(judge_fn, prompt, response_a, response_b):
    r_ab = judge_fn(prompt, response_a, response_b)
    r_ba = judge_fn(prompt, response_b, response_a)
    swap = {'a': 'b', 'b': 'a', 'tie': 'tie'}
    winner_ba_norm = swap[r_ba.winner]
    consistent = r_ab.winner == winner_ba_norm
    return {
        'order_ab': r_ab.winner,
        'order_ba_normalized': winner_ba_norm,
        'consistent': consistent,
        'positional_flip': not consistent,
    }

bias = detect_positional_bias(judge, prompt, short_correct, long_verbose)
print('Positional bias check:')
for k, v in bias.items():
    print(f'  {k}: {v}')
if bias['positional_flip']:
    print('\nWARNING: judge flipped verdict on position alone')

In [ ]:
import re

def parse_judge(raw):
    score_match = re.search(r'(?:score|rating)[:\s]+([0-9](?:\.[0-9])?)/10', raw, re.I)
    winner_match = re.search(r'(?:winner|prefer|better)[:\s]+(?:response\s+)?([AB])', raw, re.I)
    score = float(score_match.group(1)) if score_match else None
    winner = winner_match.group(1).lower() if winner_match else None
    return {'score': score, 'winner': winner, 'raw': raw[:200]}

example_output = 'Response A is clear and accurate. Score: 8/10. Winner: Response A'
print(parse_judge(example_output))

## Rubric Design

Effective judge rubrics:
1. **Decompose quality into orthogonal dimensions** — factual accuracy, completeness, clarity, and safety scored separately
2. **Provide anchor examples** — what does a 3/10 vs 8/10 factual accuracy look like?
3. **Require chain-of-thought before scoring** — reasoning before scoring improves calibration
4. **Specify the failure mode** — a safety rubric looks completely different from a helpfulness rubric

## Production Considerations

At scale, LLM-as-judge is used for:
- **Post-deployment monitoring**: sample 1-5% of production traffic, run judge evaluation, alert on quality regression
- **A/B test evaluation**: replace human preference collection for faster iteration
- **Training data filtering**: filter synthetic datasets before fine-tuning
- **RLHF reward modeling**: use judge scores when human preferences are sparse

**Judge drift** is the key failure mode at scale: if you update the judge model, historical scores become incomparable. Maintain frozen judge model versions with explicit versioning.